# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [2]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [3]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai google-genai --quiet
print("Done.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.6 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-monitoring 2.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google

In [4]:
# @title User input needed.

# Topic: a home coffee brewing guide. Each document has at least 4 sentences.
DOCUMENTS = {
    "pour_over": (
        "Pour-over coffee uses a medium grind, similar in texture to table salt. "
        "Use a ratio of 1 gram of coffee to 16 grams of water. "
        "Heat the water to between 92 and 96 degrees Celsius. "
        "Start with a 30 to 45 second bloom, using about twice the coffee's weight in water. "
        "The total brew time should be 3 to 4 minutes."
    ),
    "french_press": (
        "French press coffee uses a coarse grind, similar in texture to sea salt. "
        "Use a ratio of 1 gram of coffee to 15 grams of water. "
        "Heat the water to between 93 and 96 degrees Celsius. "
        "Steep the coffee for 4 minutes before pressing the plunger down slowly. "
        "Pour the coffee out straight after pressing so that it does not over-extract."
    ),
    "espresso": (
        "Espresso uses a fine grind, slightly finer than table salt. "
        "A standard double shot uses 18 grams of coffee to produce about 36 grams of liquid, a ratio of 1 to 2. "
        "The machine pushes water through the coffee at about 9 bars of pressure. "
        "A good shot takes 25 to 30 seconds to pull. "
        "If the shot runs too fast, grind finer, and if it runs too slowly, grind coarser."
    ),
    "cold_brew": (
        "Cold brew uses a coarse grind and water at room temperature or colder. "
        "Use a ratio of 1 gram of coffee to 8 grams of water to make a concentrate. "
        "Steep the coffee for 12 to 18 hours before straining it. "
        "Dilute the concentrate with an equal amount of water or milk before drinking. "
        "Keep the strained concentrate in the fridge and finish it within 7 days."
    ),
    "bean_storage": (
        "Store coffee beans in an airtight, opaque container in a cool, dark place. "
        "Avoid keeping opened beans in the fridge, because moisture and food odours can spoil them. "
        "Beans taste best within 2 to 4 weeks of the roast date. "
        "Grind beans just before brewing, because ground coffee goes stale much faster than whole beans."
    ),
}


In [5]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for keyword match.
print(search_documents("french press steep time"))


[Source: french_press]
French press coffee uses a coarse grind, similar in texture to sea salt. Use a ratio of 1 gram of coffee to 15 grams of water. Heat the water to between 93 and 96 degrees Celsius. Steep the coffee for 4 minutes before pressing the plunger down slowly. Pour the coffee out straight after pressing so that it does not over-extract.

[Source: pour_over]
Pour-over coffee uses a medium grind, similar in texture to table salt. Use a ratio of 1 gram of coffee to 16 grams of water. Heat the water to between 92 and 96 degrees Celsius. Start with a 30 to 45 second bloom, using about twice the coffee's weight in water. The total brew time should be 3 to 4 minutes.


In [6]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# Shared with the agent loop below, so the rules apply on every step.
SYSTEM_PROMPT = '''
You are a barista trainer who follows the ReAct pattern: reason, act, observe, repeat.
Before each tool call, write one short line starting with "Thought:" that says what
you need to find out and why.
When you need specific information about grind size, ratios, temperatures, brew times
or storing coffee, use the search_documents tool.
If a question covers more than one topic, search for each topic separately.
Only answer based on what you find in the documents.
If the documents do not contain the answer, say so clearly.
'''

model = genai.GenerativeModel(
    # Current free-tier model from https://ai.google.dev/gemini-api/docs/models
    model_name="gemini-3.5-flash-lite",
    tools=[search_documents],
    system_instruction=SYSTEM_PROMPT,
)

print("Model ready:", model.model_name)


Model ready: models/gemini-3.5-flash-lite


/tmp/ipykernel_3132/3340755499.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [7]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        parts = model_content.parts or []
        calls = [p.function_call for p in parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Show the model's reasoning before it acts
        for p in parts:
            if p.text and not p.thought:
                print(f"Step {step}: {p.text.strip()}")

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("How long should a French press steep, and what grind size does it use?")

Question: How long should a French press steep, and what grind size does it use?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'French press steep time grind size'})
  Result preview: [Source: french_press]
French press coffee uses a coarse grind, similar in texture to sea salt. Use ...

Final answer:
A French press should steep for **4 minutes** before you press the plunger down slowly. It uses a **coarse grind**, similar in texture to sea salt.


In [8]:
# @title Ask follow up questions

questions = [
    # This question needs two searches, one for each brewing method.
    "Which uses a finer grind and a shorter brew time, espresso or cold brew? Give the numbers for both.",
    # This question needs a search and then a calculation.
    "I have 30 grams of coffee. How much water do I need for a pour-over?",
    # This question combines two facts from one document.
    "Should I keep my opened beans in the fridge, and when should I grind them?",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: Which uses a finer grind and a shorter brew time, espresso or cold brew? Give the numbers for both.
--------------------------------------------------
Step 1: model calls search_documents({'query': 'espresso grind brew time'})
  Result preview: [Source: pour_over]
Pour-over coffee uses a medium grind, similar in texture to table salt. Use a ra...
Step 2: model calls search_documents({'query': 'cold brew grind brew time'})
  Result preview: [Source: pour_over]
Pour-over coffee uses a medium grind, similar in texture to table salt. Use a ra...

Final answer:
Espresso uses a finer grind and a shorter brew time than cold brew. 

Here are the numbers for both:

* **Espresso:** Uses a fine grind (slightly finer than table salt) with a brew time of **25 to 30 seconds**.
* **Cold Brew:** Uses a coarse grind with a brew time of **12 to 18 hours**.

Question: I have 30 grams of coffee. How much water do I need for a pour-over?
--------------------------------------------------
Step 1:

In [9]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)